<a href="https://colab.research.google.com/github/ruudtje21/ai-sql-assistant/blob/main/yolov8_and_rcnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## INSTAL LIBRARY

In [ ]:
!pip install ultralytics --quiet
!pip install pyyaml==5.1 --quiet
!pip install 'git+https://github.com/facebookresearch/detectron2.git' --quiet
print("library terinstall!")

## IMPORT LIBRARY

In [ ]:
# Sistem & file management
import os
import shutil
import random
import json

# Komputasi & data
import numpy as np
import pandas as pd
from scipy import stats

# Computer vision
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# YOLOv8
from ultralytics import YOLO

# Detectron2 / Faster R-CNN
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.data.datasets import register_coco_instances

print("✅ Library diimport!")

## DEFINISI PATH

In [ ]:
base        = '/content/datasets/corrosion_detect'
images_path = f'{base}/images'
labels_path = f'{base}/labels'
img_train   = f'{base}/images/train'
img_val     = f'{base}/images/val'
lbl_train   = f'{base}/labels/train'
lbl_val     = f'{base}/labels/val'

print("✅ Path sudah didefinisikan!")
print(f"Base      : {base}")
print(f"Images    : {images_path}")
print(f"Labels    : {labels_path}")
print(f"Img Train : {img_train}")
print(f"Img Val   : {img_val}")
print(f"Lbl Train : {lbl_train}")
print(f"Lbl Val   : {lbl_val}")

##SETUP KAGGLE

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle siap digunakan!")

##DOWNLOAD DATASET

In [ ]:
!kaggle datasets download -d wednesday233/corrosion-detect-dataset
!unzip -q corrosion-detect-dataset.zip -d datasets/
print("✅ Dataset berhasil didownload!")

##SPLIT DATASET

In [ ]:
# Buat folder train/val
for split in ['train', 'val']:
    os.makedirs(os.path.join(images_path, split), exist_ok=True)
    os.makedirs(os.path.join(labels_path, split), exist_ok=True)

# Split 80/20
all_images = [f for f in os.listdir(images_path) if f.endswith(('.jpg','.jpeg','.png'))]
random.seed(42)
random.shuffle(all_images)
split_idx  = int(len(all_images) * 0.8)
train_imgs = all_images[:split_idx]
val_imgs   = all_images[split_idx:]

def move_files(file_list, split):
    for img_file in file_list:
        name    = os.path.splitext(img_file)[0]
        src_img = os.path.join(images_path, img_file)
        src_lbl = os.path.join(labels_path, name + '.txt')
        dst_img = os.path.join(images_path, split, img_file)
        dst_lbl = os.path.join(labels_path, split, name + '.txt')
        if os.path.exists(src_img):
            shutil.move(src_img, dst_img)
        if os.path.exists(src_lbl):
            shutil.move(src_lbl, dst_lbl)

move_files(train_imgs, 'train')
move_files(val_imgs,   'val')
print(f"✅ Split selesai! Train: {len(train_imgs)} | Val: {len(val_imgs)}")

##AUGMENTASI CLAHE DAN GAMMA

In [ ]:
for aug in ['clahe', 'gamma']:
    os.makedirs(f'{images_path}/train_{aug}', exist_ok=True)
    os.makedirs(f'{labels_path}/train_{aug}', exist_ok=True)

def apply_clahe(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)

def apply_gamma(img_bgr, gamma=1.5):
    inv_gamma = 1.0 / gamma
    table = np.array([((i/255.0)**inv_gamma)*255 for i in range(256)]).astype(np.uint8)
    return cv2.LUT(img_bgr, table)

for fname in os.listdir(img_train):
    if not fname.lower().endswith(('.jpg','.jpeg','.png')):
        continue
    name    = os.path.splitext(fname)[0]
    src_img = os.path.join(img_train, fname)
    src_lbl = os.path.join(lbl_train, name + '.txt')
    img_bgr = cv2.imread(src_img)
    cv2.imwrite(f'{images_path}/train_clahe/{fname}', apply_clahe(img_bgr))
    cv2.imwrite(f'{images_path}/train_gamma/{fname}', apply_gamma(img_bgr))
    if os.path.exists(src_lbl):
        shutil.copy(src_lbl, f'{labels_path}/train_clahe/{name}.txt')
        shutil.copy(src_lbl, f'{labels_path}/train_gamma/{name}.txt')

print(f"✅ CLAHE : {len(os.listdir(f'{images_path}/train_clahe'))} gambar")
print(f"✅ Gamma : {len(os.listdir(f'{images_path}/train_gamma'))} gambar")

##DEGRADASI TEST SET

In [ ]:
test_ori_dir = img_val
log          = []

levels = [
    (0.85, 0.90, 'deg_mild'),
    (0.75, 0.80, 'deg_moderate'),
]

for _, _, ln in levels:
    os.makedirs(f'{images_path}/{ln}', exist_ok=True)

for fname in os.listdir(test_ori_dir):
    if not fname.lower().endswith(('.jpg','.jpeg','.png')):
        continue
    fpath   = os.path.join(test_ori_dir, fname)
    img_bgr = cv2.imread(fpath)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(float)
    Y_ori     = 0.299*img_rgb[:,:,0] + 0.587*img_rgb[:,:,1] + 0.114*img_rgb[:,:,2]
    mean_awal = float(Y_ori.mean())
    std_awal  = float(Y_ori.std())
    for bf, cf, ln in levels:
        deg    = img_rgb.copy() * bf
        mean_d = deg.mean(axis=(0,1), keepdims=True)
        deg    = (deg - mean_d) * cf + mean_d
        deg    = np.clip(deg, 0, 255)
        Y_deg    = 0.299*deg[:,:,0] + 0.587*deg[:,:,1] + 0.114*deg[:,:,2]
        mean_deg = float(Y_deg.mean())
        std_deg  = float(Y_deg.std())
        alasan = []
        if mean_deg < 0.7 * mean_awal:
            alasan.append(f"mean {mean_deg:.1f} < {0.7*mean_awal:.1f}")
        if std_deg < 0.6 * std_awal:
            alasan.append(f"std {std_deg:.1f} < {0.6*std_awal:.1f}")
        out_path = f'{images_path}/{ln}/{fname}'
        if alasan:
            shutil.copy(fpath, out_path)
            status = 'SKIP'
        else:
            cv2.imwrite(out_path, cv2.cvtColor(deg.astype(np.uint8), cv2.COLOR_RGB2BGR))
            status = 'OK'
        log.append({'file': fname, 'level': ln, 'status': status})

df_log = pd.DataFrame(log)
print(df_log.groupby(['level','status']).size())
print("✅ Degradasi selesai!")

##BUAT DATA.YAML

In [ ]:
import yaml

data_yaml = {
    'path': base,
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': ['corrosion']
}

data_yaml_clahe = {
    'path': base,
    'train': 'images/train_clahe',
    'val':   'images/val',
    'nc':    1,
    'names': ['corrosion']
}

data_yaml_gamma = {
    'path': base,
    'train': 'images/train_gamma',
    'val':   'images/val',
    'nc':    1,
    'names': ['corrosion']
}

with open(f'{base}/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)
with open(f'{base}/data_clahe.yaml', 'w') as f:
    yaml.dump(data_yaml_clahe, f)
with open(f'{base}/data_gamma.yaml', 'w') as f:
    yaml.dump(data_yaml_gamma, f)

print("✅ data.yaml berhasil dibuat!")
print(f"✅ data_clahe.yaml berhasil dibuat!")
print(f"✅ data_gamma.yaml berhasil dibuat!")

##TRAINING YOLOv8

In [ ]:
# Hapus gambar tanpa label dari semua folder train DAN val
folders = [
    (img_train, lbl_train),
    (img_val, lbl_val),
    (f'{images_path}/train_clahe', f'{labels_path}/train_clahe'),
    (f'{images_path}/train_gamma', f'{labels_path}/train_gamma'),
]

for img_dir, lbl_dir in folders:
    removed = 0
    for fname in os.listdir(img_dir):
        if not fname.lower().endswith(('.jpg','.jpeg','.png')):
            continue
        name     = os.path.splitext(fname)[0]
        lbl_path = os.path.join(lbl_dir, name + '.txt')
        if not os.path.exists(lbl_path) or os.path.getsize(lbl_path) == 0:
            os.remove(os.path.join(img_dir, fname))
            removed += 1
    print(f"✅ {img_dir.split('/')[-1]} → {removed} gambar dihapus | sisa: {len(os.listdir(img_dir))} gambar")

In [ ]:
# Fix ulang semua label di semua folder
folders = [
    lbl_train,
    lbl_val,
    f'{labels_path}/train_clahe',
    f'{labels_path}/train_gamma',
]

fixed = 0
for folder in folders:
    for txt_file in os.listdir(folder):
        if not txt_file.endswith('.txt'):
            continue
        filepath = os.path.join(folder, txt_file)
        with open(filepath, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if parts:
                parts[0] = '0'
                new_lines.append(' '.join(parts) + '\n')
        with open(filepath, 'w') as f:
            f.writelines(new_lines)
        fixed += 1

print(f"✅ {fixed} file label diperbaiki!")

In [ ]:
models_yolo = {
    'yolo_A': f'{base}/data.yaml',
    'yolo_B': f'{base}/data_clahe.yaml',
    'yolo_C': f'{base}/data_gamma.yaml',
}

for name, yaml_path in models_yolo.items():
    print(f"\n{'='*40}")
    print(f"Training {name}...")
    print(f"{'='*40}")
    model = YOLO('yolov8n.pt')
    model.train(
        data    = yaml_path,
        epochs  = 50,
        imgsz   = 640,
        batch   = 16,
        name    = name,
        project = '/content/runs/yolo',
        exist_ok= True,
        verbose = False
    )
    print(f"✅ {name} selesai!")

print("\n✅ Semua model YOLO selesai ditraining!")

In [ ]:
print(f"Train label       : {len(os.listdir(lbl_train))} file")
print(f"Val label         : {len(os.listdir(lbl_val))} file")
print(f"Train clahe label : {len(os.listdir(f'{labels_path}/train_clahe'))} file")
print(f"Train gamma label : {len(os.listdir(f'{labels_path}/train_gamma'))} file")

In [ ]:
import os

for model in ['yolo_A', 'yolo_B', 'yolo_C']:
    path = f'/content/runs/yolo/{model}/weights'
    if os.path.exists(path):
        print(f"✅ {model} → {os.listdir(path)}")
    else:
        print(f"❌ {model} → folder tidak ditemukan!")

##EVALUASI

In [ ]:
test_sets = {
    'original':     img_val,
    'deg_mild':     f'{images_path}/deg_mild',
    'deg_moderate': f'{images_path}/deg_moderate',
}

results_log_yolo = []

for model_name in ['yolo_A', 'yolo_B', 'yolo_C']:
    model = YOLO(f'/content/runs/yolo/{model_name}/weights/best.pt')
    for test_name, test_path in test_sets.items():
        print(f"Evaluasi {model_name} pada {test_name}...")
        for fname in os.listdir(test_path):
            if not fname.lower().endswith(('.jpg','.jpeg','.png')):
                continue
            img_path = os.path.join(test_path, fname)
            img_bgr  = cv2.imread(img_path)
            img_h, img_w = img_bgr.shape[:2]
            results  = model(img_path, verbose=False)
            boxes    = results[0].boxes
            n_instance = len(boxes)
            if n_instance > 0:
                areas = []
                for box in boxes.xyxy.cpu().numpy():
                    x1, y1, x2, y2 = box
                    bw = (x2 - x1) / img_w
                    bh = (y2 - y1) / img_h
                    areas.append(bw * bh)
                area_ratio  = sum(areas)
                box_density = n_instance / (img_w * img_h) * 1e6
            else:
                area_ratio  = 0.0
                box_density = 0.0
            results_log_yolo.append({
                'model':       model_name,
                'test_set':    test_name,
                'file':        fname,
                'n_instance':  n_instance,
                'area_ratio':  round(area_ratio, 4),
                'box_density': round(box_density, 4),
            })

df_results_yolo = pd.DataFrame(results_log_yolo)
df_results_yolo.to_csv('eval_results_yolo.csv', index=False)
print("\n✅ Evaluasi YOLO selesai!")
print(df_results_yolo.groupby(['model','test_set'])['n_instance'].mean().round(2))

##HITUNG STABILITAS YOLO

In [ ]:
df_yolo = pd.read_csv('eval_results_yolo.csv')

metrics_cols  = ['n_instance', 'area_ratio', 'box_density']
test_order    = ['original', 'deg_mild', 'deg_moderate']
stability_log = []

for model_name in df_yolo['model'].unique():
    df_model = df_yolo[df_yolo['model'] == model_name]
    for metric in metrics_cols:
        means = {}
        for ts in test_order:
            means[ts] = df_model[df_model['test_set']==ts][metric].mean()
        vals = [means[ts] for ts in test_order]
        cv = (np.std(vals) / np.mean(vals) * 100) if np.mean(vals) != 0 else 0
        if means['original'] != 0:
            rel_change = (means['deg_moderate'] - means['original']) / means['original'] * 100
        else:
            rel_change = 0
        deg_levels = [0, 1, 2]
        spearman_r, spearman_p = stats.spearmanr(deg_levels, vals)
        stability_log.append({
            'model':         model_name,
            'metric':        metric,
            'mean_ori':      round(means['original'], 4),
            'mean_mild':     round(means['deg_mild'], 4),
            'mean_mod':      round(means['deg_moderate'], 4),
            'CV(%)':         round(cv, 2),
            'rel_change(%)': round(rel_change, 2),
            'spearman_r':    round(spearman_r, 3),
            'spearman_p':    round(spearman_p, 3),
        })

df_stability_yolo = pd.DataFrame(stability_log)
df_stability_yolo.to_csv('stability_results.csv', index=False)

for metric in metrics_cols:
    print("\n" + "="*55)
    print("Metrik: " + metric)
    print("="*55)
    sub = df_stability_yolo[df_stability_yolo['metric']==metric]
    print(sub[['model','CV(%)','rel_change(%)','spearman_r']].to_string(index=False))

print("\nDisimpan ke stability_results.csv")

##HITUNG RANKING STABILITY

In [ ]:
df_yolo = pd.read_csv('eval_results_yolo.csv')

ranking_log = []

for model_name in df_yolo['model'].unique():
    df_model = df_yolo[df_yolo['model'] == model_name]
    for metric in metrics_cols:
        ranks = {}
        for ts in test_order:
            vals = df_model[df_model['test_set']==ts].set_index('file')[metric]
            ranks[ts] = vals.rank(ascending=False)
        common_files = set(ranks['original'].index)
        for ts in test_order[1:]:
            common_files &= set(ranks[ts].index)
        common_files = list(common_files)
        r1, _ = stats.spearmanr(
            ranks['original'][common_files],
            ranks['deg_mild'][common_files]
        )
        r2, _ = stats.spearmanr(
            ranks['original'][common_files],
            ranks['deg_moderate'][common_files]
        )
        ranking_log.append({
            'model':             model_name,
            'metric':            metric,
            'spearman_ori_mild': round(r1, 3),
            'spearman_ori_mod':  round(r2, 3),
            'rank_stable':       'YA' if r2 > 0.7 else 'TIDAK'
        })

df_ranking_yolo = pd.DataFrame(ranking_log)
df_ranking_yolo.to_csv('ranking_stability.csv', index=False)

for metric in metrics_cols:
    print("\n" + "="*55)
    print("Ranking Stability: " + metric)
    print("="*55)
    sub = df_ranking_yolo[df_ranking_yolo['metric']==metric]
    print(sub[['model','spearman_ori_mild','spearman_ori_mod','rank_stable']].to_string(index=False))

print("\nDisimpan ke ranking_stability.csv")

##REKAP HASIL YOLO

In [ ]:
df_s = pd.read_csv('stability_results.csv')
df_r = pd.read_csv('ranking_stability.csv')

rows = []
for model in ['yolo_A', 'yolo_B', 'yolo_C']:
    for metric in ['n_instance', 'area_ratio', 'box_density']:
        s = df_s[(df_s['model']==model) & (df_s['metric']==metric)].iloc[0]
        r = df_r[(df_r['model']==model) & (df_r['metric']==metric)].iloc[0]
        rows.append({
            'Model':             model,
            'Metrik':            metric,
            'Mean_Ori':          s['mean_ori'],
            'Mean_Mild':         s['mean_mild'],
            'Mean_Moderate':     s['mean_mod'],
            'CV_pct':            s['CV(%)'],
            'RelChange_pct':     s['rel_change(%)'],
            'Spearman_trend':    s['spearman_r'],
            'Spearman_Ori_Mild': r['spearman_ori_mild'],
            'Spearman_Ori_Mod':  r['spearman_ori_mod'],
            'Rank_Stabil':       r['rank_stable'],
        })

df_rekap_yolo = pd.DataFrame(rows)
df_rekap_yolo.to_csv('rekap_akhir_yolo.csv', index=False)

for metric in ['n_instance', 'area_ratio', 'box_density']:
    print("\n" + "="*70)
    print("Metrik: " + metric)
    print("="*70)
    sub = df_rekap_yolo[df_rekap_yolo['Metrik']==metric]
    print(sub[['Model','Mean_Ori','Mean_Mild','Mean_Moderate',
               'CV_pct','RelChange_pct','Spearman_trend',
               'Spearman_Ori_Mod','Rank_Stabil']].to_string(index=False))

print("\nDisimpan ke rekap_akhir_yolo.csv")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

for model in ['yolo_A', 'yolo_B', 'yolo_C']:
    path = f'/content/runs/yolo/{model}'

    # Cek file yang tersedia
    files = os.listdir(path)
    print(f"\n📁 {model}: {files}")

    # Tampilkan hasil validasi
    val_imgs = [f for f in files if f.endswith('.png') or f.endswith('.jpg')]

    for img_file in val_imgs:
        img_path = os.path.join(path, img_file)
        img = mpimg.imread(img_path)
        plt.figure(figsize=(15, 8))
        plt.imshow(img)
        plt.title(f'{model} - {img_file}', fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

In [ ]:
for model in ['yolo_A', 'yolo_B', 'yolo_C']:
    path = f'/content/runs/yolo/{model}'
    pred_imgs = [f for f in os.listdir(path) if 'pred' in f]
    for img_file in pred_imgs:
        img = mpimg.imread(os.path.join(path, img_file))
        plt.figure(figsize=(15, 8))
        plt.imshow(img)
        plt.title(f'{model} - {img_file}', fontweight='bold')
        plt.axis('off')
        plt.show()

## RCNN

In [ ]:
register_coco_instances("corrosion_train",       {}, f"{base}/train.json",       img_train)
register_coco_instances("corrosion_train_clahe", {}, f"{base}/train_clahe.json", f"{images_path}/train_clahe")
register_coco_instances("corrosion_train_gamma", {}, f"{base}/train_gamma.json", f"{images_path}/train_gamma")
register_coco_instances("corrosion_val",         {}, f"{base}/val.json",         img_val)

print("✅ Dataset terdaftar!")

##TRAINING RCNN (3000 ITERASI)

In [ ]:
def train_rcnn(train_dataset, output_dir, num_iter=3000):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(
        "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
    cfg.DATASETS.TRAIN = (train_dataset,)
    cfg.DATASETS.TEST  = ()
    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
        "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")
    cfg.SOLVER.IMS_PER_BATCH     = 2
    cfg.SOLVER.BASE_LR           = 0.002
    cfg.SOLVER.MAX_ITER          = num_iter
    cfg.SOLVER.STEPS             = (2000, 2500)
    cfg.SOLVER.GAMMA             = 0.1
    cfg.SOLVER.WARMUP_ITERS      = 500
    cfg.SOLVER.CHECKPOINT_PERIOD = num_iter
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256
    cfg.MODEL.ROI_HEADS.NUM_CLASSES          = 1
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST    = 0.3
    cfg.OUTPUT_DIR = output_dir
    os.makedirs(output_dir, exist_ok=True)
    trainer = DefaultTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    print(f"✅ {output_dir} selesai!")
    return cfg

for name, dataset in [
    ('rcnn_A', 'corrosion_train'),
    ('rcnn_B', 'corrosion_train_clahe'),
    ('rcnn_C', 'corrosion_train_gamma'),
]:
    print(f"\n{'='*40}")
    print(f"Training {name}...")
    print(f"{'='*40}")
    train_rcnn(dataset, f'/content/runs/rcnn/{name}')

print("\n✅ Semua model RCNN selesai ditraining!")

##CEK METRIK TRAINING RCNN

In [ ]:
for model in ['rcnn_A', 'rcnn_B', 'rcnn_C']:
    with open(f'/content/runs/rcnn/{model}/metrics.json', 'r') as f:
        lines = f.readlines()
    last = json.loads(lines[-1])
    print(f"{model} → total_loss: {last['total_loss']:.4f} | fg_cls_accuracy: {last['fast_rcnn/fg_cls_accuracy']:.4f}")

##DEFINISI FUNGSI GET_PREDICTOR DAN VISUALIZE_PREDICTIONS

In [ ]:
def get_predictor(weights_path, threshold=0.3):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(
        "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg.MODEL.WEIGHTS = weights_path
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = threshold
    return DefaultPredictor(cfg)

def visualize_predictions(predictor, images_dir, title, n=8):
    all_files = [f for f in os.listdir(images_dir)
                 if f.endswith(('.jpg','.jpeg','.png'))]
    files = random.sample(all_files, min(n, len(all_files)))
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    for i, fname in enumerate(files):
        img_path = os.path.join(images_dir, fname)
        img_bgr  = cv2.imread(img_path)
        img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        outputs  = predictor(img_bgr)
        boxes    = outputs['instances'].pred_boxes.tensor.cpu().numpy()
        scores   = outputs['instances'].scores.cpu().numpy()
        axes[i].imshow(img_rgb)
        for box, score in zip(boxes, scores):
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor='red', facecolor='none'
            )
            axes[i].add_patch(rect)
            axes[i].text(x1, y1-5, f'{score:.2f}',
                        color='yellow', fontsize=8, fontweight='bold',
                        bbox=dict(facecolor='red', alpha=0.5, pad=1))
        axes[i].set_title(f'{fname[:15]}\n{len(boxes)} deteksi', fontsize=8)
        axes[i].axis('off')
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_name = title.replace(' ', '_') + '.png'
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()
    print("Disimpan ke " + save_name)

print("✅ Fungsi siap!")

##VISUALISASI DAN LOAD PREDICTOR

In [ ]:
predictor_a = get_predictor('/content/runs/rcnn/rcnn_A/model_final.pth')
predictor_b = get_predictor('/content/runs/rcnn/rcnn_B/model_final.pth')
predictor_c = get_predictor('/content/runs/rcnn/rcnn_C/model_final.pth')

visualize_predictions(predictor_a, img_val, 'RCNN_A - Dataset Asli')
visualize_predictions(predictor_b, img_val, 'RCNN_B - CLAHE Augmented')
visualize_predictions(predictor_c, img_val, 'RCNN_C - Gamma Augmented')

##EVALUASI RCNN PADA TEST SET TERDEGRADASI

In [ ]:
test_sets = {
    'original':     img_val,
    'deg_mild':     f'{images_path}/deg_mild',
    'deg_moderate': f'{images_path}/deg_moderate',
}

models_rcnn = {
    'rcnn_A': predictor_a,
    'rcnn_B': predictor_b,
    'rcnn_C': predictor_c,
}

results_log_rcnn = []

for model_name, predictor in models_rcnn.items():
    for test_name, test_path in test_sets.items():
        print(f"Evaluasi {model_name} pada {test_name}...")
        for fname in os.listdir(test_path):
            if not fname.lower().endswith(('.jpg','.jpeg','.png')):
                continue
            img_path = os.path.join(test_path, fname)
            img_bgr  = cv2.imread(img_path)
            outputs  = predictor(img_bgr)
            boxes    = outputs['instances'].pred_boxes.tensor.cpu().numpy()
            scores   = outputs['instances'].scores.cpu().numpy()
            n_instance = len(boxes)
            img_h, img_w = img_bgr.shape[:2]
            if n_instance > 0:
                areas = []
                for box in boxes:
                    x1, y1, x2, y2 = box
                    bw = (x2 - x1) / img_w
                    bh = (y2 - y1) / img_h
                    areas.append(bw * bh)
                area_ratio  = sum(areas)
                box_density = n_instance / (img_w * img_h) * 1e6
            else:
                area_ratio  = 0.0
                box_density = 0.0
            results_log_rcnn.append({
                'model':       model_name,
                'test_set':    test_name,
                'file':        fname,
                'n_instance':  n_instance,
                'area_ratio':  round(area_ratio, 4),
                'box_density': round(box_density, 4),
            })

df_results_rcnn = pd.DataFrame(results_log_rcnn)
df_results_rcnn.to_csv('eval_results_rcnn.csv', index=False)
print("\n✅ Evaluasi RCNN selesai!")
print(df_results_rcnn.groupby(['model','test_set'])['n_instance'].mean().round(2))

##HITUNG STABILITAS RCNN

In [ ]:
df_rcnn = pd.read_csv('eval_results_rcnn.csv')

stability_log = []

for model_name in df_rcnn['model'].unique():
    df_model = df_rcnn[df_rcnn['model'] == model_name]
    for metric in metrics_cols:
        means = {}
        for ts in test_order:
            means[ts] = df_model[df_model['test_set']==ts][metric].mean()
        vals = [means[ts] for ts in test_order]
        cv = (np.std(vals) / np.mean(vals) * 100) if np.mean(vals) != 0 else 0
        if means['original'] != 0:
            rel_change = (means['deg_moderate'] - means['original']) / means['original'] * 100
        else:
            rel_change = 0
        deg_levels = [0, 1, 2]
        spearman_r, spearman_p = stats.spearmanr(deg_levels, vals)
        stability_log.append({
            'model':         model_name,
            'metric':        metric,
            'mean_ori':      round(means['original'], 4),
            'mean_mild':     round(means['deg_mild'], 4),
            'mean_mod':      round(means['deg_moderate'], 4),
            'CV(%)':         round(cv, 2),
            'rel_change(%)': round(rel_change, 2),
            'spearman_r':    round(spearman_r, 3),
            'spearman_p':    round(spearman_p, 3),
        })

df_stability_rcnn = pd.DataFrame(stability_log)
df_stability_rcnn.to_csv('stability_results_rcnn.csv', index=False)

for metric in metrics_cols:
    print("\n" + "="*55)
    print("Metrik: " + metric)
    print("="*55)
    sub = df_stability_rcnn[df_stability_rcnn['metric']==metric]
    print(sub[['model','CV(%)','rel_change(%)','spearman_r']].to_string(index=False))

print("\nDisimpan ke stability_results_rcnn.csv")

##HITUNG RANKING STABILITY RCNN

In [ ]:
df_rcnn = pd.read_csv('eval_results_rcnn.csv')

ranking_log = []

for model_name in df_rcnn['model'].unique():
    df_model = df_rcnn[df_rcnn['model'] == model_name]
    for metric in metrics_cols:
        ranks = {}
        for ts in test_order:
            vals = df_model[df_model['test_set']==ts].set_index('file')[metric]
            ranks[ts] = vals.rank(ascending=False)
        common_files = set(ranks['original'].index)
        for ts in test_order[1:]:
            common_files &= set(ranks[ts].index)
        common_files = list(common_files)
        r1, _ = stats.spearmanr(
            ranks['original'][common_files],
            ranks['deg_mild'][common_files]
        )
        r2, _ = stats.spearmanr(
            ranks['original'][common_files],
            ranks['deg_moderate'][common_files]
        )
        ranking_log.append({
            'model':             model_name,
            'metric':            metric,
            'spearman_ori_mild': round(r1, 3),
            'spearman_ori_mod':  round(r2, 3),
            'rank_stable':       'YA' if r2 > 0.7 else 'TIDAK'
        })

df_ranking_rcnn = pd.DataFrame(ranking_log)
df_ranking_rcnn.to_csv('ranking_stability_rcnn.csv', index=False)

for metric in metrics_cols:
    print("\n" + "="*55)
    print("Ranking Stability: " + metric)
    print("="*55)
    sub = df_ranking_rcnn[df_ranking_rcnn['metric']==metric]
    print(sub[['model','spearman_ori_mild','spearman_ori_mod','rank_stable']].to_string(index=False))

print("\nDisimpan ke ranking_stability_rcnn.csv")

##REKAP HASIL RCNN

In [ ]:
df_s = pd.read_csv('stability_results_rcnn.csv')
df_r = pd.read_csv('ranking_stability_rcnn.csv')

rows = []
for model in ['rcnn_A', 'rcnn_B', 'rcnn_C']:
    for metric in ['n_instance', 'area_ratio', 'box_density']:
        s = df_s[(df_s['model']==model) & (df_s['metric']==metric)].iloc[0]
        r = df_r[(df_r['model']==model) & (df_r['metric']==metric)].iloc[0]
        rows.append({
            'Model':             model,
            'Metrik':            metric,
            'Mean_Ori':          s['mean_ori'],
            'Mean_Mild':         s['mean_mild'],
            'Mean_Moderate':     s['mean_mod'],
            'CV_pct':            s['CV(%)'],
            'RelChange_pct':     s['rel_change(%)'],
            'Spearman_trend':    s['spearman_r'],
            'Spearman_Ori_Mild': r['spearman_ori_mild'],
            'Spearman_Ori_Mod':  r['spearman_ori_mod'],
            'Rank_Stabil':       r['rank_stable'],
        })

df_rekap_rcnn = pd.DataFrame(rows)
df_rekap_rcnn.to_csv('rekap_akhir_rcnn.csv', index=False)

for metric in ['n_instance', 'area_ratio', 'box_density']:
    print("\n" + "="*70)
    print("Metrik: " + metric)
    print("="*70)
    sub = df_rekap_rcnn[df_rekap_rcnn['Metrik']==metric]
    print(sub[['Model','Mean_Ori','Mean_Mild','Mean_Moderate',
               'CV_pct','RelChange_pct','Spearman_trend',
               'Spearman_Ori_Mod','Rank_Stabil']].to_string(index=False))

print("\nDisimpan ke rekap_akhir_rcnn.csv")

##PERBANDINGAN YOLO VS FASTER RCNN (VISUALISASI)

In [ ]:
df_yolo_s = pd.read_csv('stability_results.csv')
df_rcnn_s = pd.read_csv('stability_results_rcnn.csv')
df_yolo_r = pd.read_csv('ranking_stability.csv')
df_rcnn_r = pd.read_csv('ranking_stability_rcnn.csv')

metrics     = ['n_instance', 'area_ratio', 'box_density']
models_yolo = ['yolo_A', 'yolo_B', 'yolo_C']
models_rcnn = ['rcnn_A', 'rcnn_B', 'rcnn_C']
colors_yolo = ['#2196F3', '#4CAF50', '#FF9800']
colors_rcnn = ['#1565C0', '#2E7D32', '#E65100']

x     = np.arange(len(metrics))
width = 0.15

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot 1: CV
ax = axes[0]
for i, (model, color) in enumerate(zip(models_yolo, colors_yolo)):
    vals = [df_yolo_s[(df_yolo_s['model']==model) & (df_yolo_s['metric']==m)]['CV(%)'].values[0] for m in metrics]
    ax.bar(x + i*width, vals, width, label=model, color=color, alpha=0.85)
for i, (model, color) in enumerate(zip(models_rcnn, colors_rcnn)):
    vals = [df_rcnn_s[(df_rcnn_s['model']==model) & (df_rcnn_s['metric']==m)]['CV(%)'].values[0] for m in metrics]
    ax.bar(x + (i+3)*width, vals, width, label=model, color=color, alpha=0.85)
ax.set_title('CV (%) - makin kecil makin stabil', fontweight='bold')
ax.set_xticks(x + 2.5*width)
ax.set_xticklabels(metrics, rotation=10)
ax.set_ylabel('CV (%)')
ax.legend(fontsize=7)
ax.grid(axis='y', alpha=0.3)

# Plot 2: Relative Change
ax = axes[1]
for i, (model, color) in enumerate(zip(models_yolo, colors_yolo)):
    vals = [df_yolo_s[(df_yolo_s['model']==model) & (df_yolo_s['metric']==m)]['rel_change(%)'].values[0] for m in metrics]
    ax.bar(x + i*width, vals, width, label=model, color=color, alpha=0.85)
for i, (model, color) in enumerate(zip(models_rcnn, colors_rcnn)):
    vals = [df_rcnn_s[(df_rcnn_s['model']==model) & (df_rcnn_s['metric']==m)]['rel_change(%)'].values[0] for m in metrics]
    ax.bar(x + (i+3)*width, vals, width, label=model, color=color, alpha=0.85)
ax.set_title('Relative Change % - makin kecil makin stabil', fontweight='bold')
ax.set_xticks(x + 2.5*width)
ax.set_xticklabels(metrics, rotation=10)
ax.set_ylabel('Relative Change (%)')
ax.legend(fontsize=7)
ax.grid(axis='y', alpha=0.3)
ax.axhline(0, color='black', linewidth=0.8)

# Plot 3: Ranking Stability
ax = axes[2]
for i, (model, color) in enumerate(zip(models_yolo, colors_yolo)):
    vals = [df_yolo_r[(df_yolo_r['model']==model) & (df_yolo_r['metric']==m)]['spearman_ori_mod'].values[0] for m in metrics]
    ax.bar(x + i*width, vals, width, label=model, color=color, alpha=0.85)
for i, (model, color) in enumerate(zip(models_rcnn, colors_rcnn)):
    vals = [df_rcnn_r[(df_rcnn_r['model']==model) & (df_rcnn_r['metric']==m)]['spearman_ori_mod'].values[0] for m in metrics]
    ax.bar(x + (i+3)*width, vals, width, label=model, color=color, alpha=0.85)
ax.set_title('Ranking Stability - makin tinggi makin stabil', fontweight='bold')
ax.set_xticks(x + 2.5*width)
ax.set_xticklabels(metrics, rotation=10)
ax.set_ylabel('Spearman r')
ax.legend(fontsize=7)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0.8, 1.0)
ax.axhline(0.7, color='red', linestyle='--')

plt.suptitle('Perbandingan Stabilitas YOLOv8 vs Faster R-CNN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('comparison_yolo_vs_rcnn.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Disimpan ke comparison_yolo_vs_rcnn.png")

##REKAP FINAL SEMUA MODEL

In [ ]:
df_yolo_s = pd.read_csv('stability_results.csv')
df_rcnn_s = pd.read_csv('stability_results_rcnn.csv')
df_yolo_r = pd.read_csv('ranking_stability.csv')
df_rcnn_r = pd.read_csv('ranking_stability_rcnn.csv')

rows = []
for df_s, df_r in [(df_yolo_s, df_yolo_r), (df_rcnn_s, df_rcnn_r)]:
    for model in df_s['model'].unique():
        for metric in ['n_instance', 'area_ratio', 'box_density']:
            s = df_s[(df_s['model']==model) & (df_s['metric']==metric)].iloc[0]
            r = df_r[(df_r['model']==model) & (df_r['metric']==metric)].iloc[0]
            rows.append({
                'Framework':        'YOLO' if 'yolo' in model else 'RCNN',
                'Model':            model,
                'Metrik':           metric,
                'Mean_Ori':         s['mean_ori'],
                'Mean_Mild':        s['mean_mild'],
                'Mean_Moderate':    s['mean_mod'],
                'CV_pct':           s['CV(%)'],
                'RelChange_pct':    s['rel_change(%)'],
                'Spearman_trend':   s['spearman_r'],
                'Spearman_Ori_Mod': r['spearman_ori_mod'],
                'Rank_Stabil':      r['rank_stable'],
            })

df_final = pd.DataFrame(rows)
df_final.to_csv('rekap_final_semua.csv', index=False)

for metric in ['n_instance', 'area_ratio', 'box_density']:
    print("\n" + "="*80)
    print("Metrik: " + metric)
    print("="*80)
    sub = df_final[df_final['Metrik']==metric]
    print(sub[['Framework','Model','Mean_Ori','Mean_Moderate',
               'CV_pct','RelChange_pct','Spearman_Ori_Mod',
               'Rank_Stabil']].to_string(index=False))

print("Disimpan ke rekap_final_semua.csv")

##DOWNLOAD SEMUA FILE

In [ ]:
from google.colab import files

for f in [
    'eval_results_yolo.csv',
    'stability_results.csv',
    'ranking_stability.csv',
    'rekap_akhir_yolo.csv',
    'eval_results_rcnn.csv',
    'stability_results_rcnn.csv',
    'ranking_stability_rcnn.csv',
    'rekap_akhir_rcnn.csv',
    'rekap_final_semua.csv',
    'comparison_yolo_vs_rcnn.png',
    'RCNN_A_-_Dataset_Asli.png',
    'RCNN_B_-_CLAHE_Augmented.png',
    'RCNN_C_-_Gamma_Augmented.png',
]:
    try:
        files.download(f)
        print(f"✅ {f} didownload!")
    except Exception as e:
        print(f"❌ {f} gagal: {e}")